# Week 7: Deep Learning Foundations
## Notebook 09: Long Short-Term Memory (LSTM) Cell from Scratch

### 1. Conceptual Intuition & Real-World Use Case
Vanilla Recurrent Neural Networks (RNNs) suffer from the **Vanishing / Exploding Gradient Problem**:
- In standard RNNs, hidden state transitions $h_t = \tanh(W h_{t-1} + U x_t)$ involve continuous matrix multiplications by $W$.
- Over 20+ time steps, repeated multiplication by eigenvalues $< 1$ drives gradients to 0, completely erasing long-term memory.

In 1997, Hochreiter & Schmidhuber invented the **LSTM (Long Short-Term Memory)**:
- Introduces an uninterrupted **Cell State ($C_t$)** acting as an information highway.
- Regulates memory flow through three differentiable **Gating Mechanisms**:
  1. **Forget Gate ($f_t$):** What proportion of past memory to discard.
  2. **Input Gate ($i_t$):** What new candidate information to store.
  3. **Output Gate ($o_t$):** What hidden state information to emit to the next layer.

---

### 2. Mathematical Foundations: LSTM Equations
At time step $t$ given input $x_t$, previous hidden state $h_{t-1}$, and previous cell state $C_{t-1}$:

1. **Forget Gate:**
   $$f_t = \sigma(W_f x_t + U_f h_{t-1} + b_f)$$
2. **Input Gate & Candidate Memory:**
   $$i_t = \sigma(W_i x_t + U_i h_{t-1} + b_i)$$
   $$\tilde{C}_t = \tanh(W_c x_t + U_c h_{t-1} + b_c)$$
3. **Cell State Update (Linear Constant Error Carousel):**
   $$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$
   *(Notice additive update! Gradients can flow backward without vanishing exponential decay!)*
4. **Output Gate & Hidden State:**
   $$o_t = \sigma(W_o x_t + U_o h_{t-1} + b_o)$$
   $$h_t = o_t \odot \tanh(C_t)$$

In [ ]:
import numpy as np

np.random.seed(42)

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -15, 15)))

class LSTMCellScratch:
    def __init__(self, input_dim, hidden_dim):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        
        # Concatenated weights for all 4 gates [f, i, C_tilde, o]
        # Total gate dimensions: 4 * hidden_dim
        concat_dim = input_dim + hidden_dim
        self.W = np.random.randn(concat_dim, 4 * hidden_dim) * 0.1
        self.b = np.zeros(4 * hidden_dim)
        
        # Forget gate bias initialized to +1.0 (Jozefowicz et al. best practice!)
        self.b[:hidden_dim] = 1.0

    def forward_step(self, x_t, h_prev, C_prev):
        H = self.hidden_dim
        # Concatenate input x_t and previous hidden state h_{t-1}
        combined = np.concatenate([x_t, h_prev])
        
        # Affine transform for all 4 gates in a single BLAS dot product!
        gates = combined @ self.W + self.b
        
        # Slice into individual gates
        f_raw = gates[:H]
        i_raw = gates[H:2*H]
        c_raw = gates[2*H:3*H]
        o_raw = gates[3*H:]
        
        # Activations
        f_t = sigmoid(f_raw)             # Forget gate
        i_t = sigmoid(i_raw)             # Input gate
        c_tilde = np.tanh(c_raw)          # Candidate state
        o_t = sigmoid(o_raw)             # Output gate
        
        # Cell state update
        C_t = f_t * C_prev + i_t * c_tilde
        
        # Hidden state update
        h_t = o_t * np.tanh(C_t)
        
        return h_t, C_t, (f_t, i_t, c_tilde, o_t)

# Step 2: Run Forward Pass across a 5-step sequence
seq_len = 5
in_dim = 3
hid_dim = 4

lstm = LSTMCellScratch(input_dim=in_dim, hidden_dim=hid_dim)
x_seq = np.random.randn(seq_len, in_dim)

h_t = np.zeros(hid_dim)
C_t = np.zeros(hid_dim)

print("--- Sequential LSTM State Progression ---")
for t in range(seq_len):
    h_t, C_t, gate_vals = lstm.forward_step(x_seq[t], h_t, C_t)
    f_t, i_t, _, o_t = gate_vals
    print(f"Step {t + 1}:")
    print(f"  Forget gate mean: {np.mean(f_t):.3f} | Input gate mean: {np.mean(i_t):.3f}")
    print(f"  Cell State C_t:   {np.round(C_t, 3)}")
    print(f"  Hidden State h_t: {np.round(h_t, 3)}")

### 3. Step-by-Step Code Walkthrough
1. **Gate Matrix Fusion:** Concatenating $[x_t, h_{t-1}]$ and projecting into $4 \times H$ hidden dimensions computes all 4 gates in a single optimized matrix multiplication.
2. **Forget Bias Initialization ($b_f = 1.0$):** Initializes forget gates close to $\sigma(1.0) \approx 0.73$, preventing the model from prematurely erasing past information at training start.
3. **Additive Constant Error Carousel:** $C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$ prevents vanishing gradients across time.

---

### 4. Industry Applications & Trade-offs
- **Speech Recognition & Sensor Processing:** LSTMs remain competitive in streaming low-latency edge audio processing.
- **Trade-off:** Sequential computation cannot be parallelized across time steps, which led to Transformers dominating text modeling.

---

### 5. Top Interview Questions & Answers
**Q1: Why is the Forget Gate bias initialized to +1.0?**
> **Answer:** If initialized to 0, $\sigma(0) = 0.5$, meaning the cell state forgets 50% of its information at every time step. Setting $b_f = 1.0$ makes $\sigma(1) \approx 0.73$, forcing the network to remember previous context by default unless trained otherwise.

**Q2: What is the primary architectural difference between an LSTM and a GRU?**
> **Answer:** GRU (Gated Recurrent Unit) eliminates the separate cell state $C_t$, merging it directly into the hidden state $h_t$. It replaces the forget and input gates with a single **Update Gate ($z_t$)** and adds a **Reset Gate ($r_t$)**, reducing parameter count by 25% for faster training.